# 1. Configuração Inicial e Imports

## Objetivo desta Seção
Importar todas as bibliotecas necessárias para a solução completa e configurar parâmetros visuais dos gráficos.

## Bibliotecas Utilizadas

### Processamento e Estruturas de Dados
- **networkx (nx)**: Criação, manipulação e análise de grafos
- **pandas (pd)**: Estruturas de dados para análise (DataFrame, Series)
- **numpy (np)**: Computação numérica e operações matemáticas

### Visualização
- **matplotlib.pyplot (plt)**: Criação de gráficos e visualizações
- **seaborn (sns)**: Estilo e temas para matplotlib

### Utilitários
- **os**: Operações do sistema operacional
- **pathlib.Path**: Manipulação de caminhos de arquivos
- **time**: Medição de tempo de execução
- **itertools.product**: Geração de combinações
- **typing**: Type hints para melhor documentação

## Configurações Visuais
- Estilo whitegrid para melhor legibilidade
- Figura padrão de 12x6 polegadas
- Tamanho de fonte padrão de 10pt

In [1]:
# Importar todas as bibliotecas necessárias
import os
import csv
import time
import json
import random
from pathlib import Path
from itertools import product
from typing import Dict, Tuple, List, Set

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configurar estilo dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Todas as bibliotecas importadas com sucesso")

✓ Todas as bibliotecas importadas com sucesso


# 2. Criação da Estrutura de Diretórios

## Objetivo desta Seção
Implementar uma classe auxiliar para criar e gerenciar automaticamente a estrutura de diretórios necessária para armazenar resultados, gráficos e visualizações de ambas as partes da solução.

## Estrutura Criada

```
resultados_2/
├── parte1/
│   ├── csv/              # Parâmetros e resultados da força bruta
│   ├── grafos/           # Visualizações dos grafos coloridos (PNG)
│   └── graficos/         # Gráficos de análise e escalabilidade (PNG)
└── parte2/
    ├── csv/              # Resultados da heurística DIMACS
    ├── grafos/           # Visualizações dos grafos coloridos (PNG)
    └── graficos/         # Gráficos comparativos (PNG)
```

## Funcionalidades da Classe ConfiguradorDiretorios

### `criar_estrutura()`
- Cria automaticamente toda a estrutura de diretórios
- Garante que todos os caminhos existem antes de salvar arquivos
- Usa `parents=True` e `exist_ok=True` para evitar erros

### `obter_caminho_arquivo(tipo_parte, tipo_saida, nome_arquivo)`
- Fornece caminhos consistentes para arquivo de saída
- Exemplo: `'resultados_2/parte1/csv/parametros_grafos.csv'`
- Reduz hardcoding de caminhos no código

In [2]:
class ConfiguradorDiretorios:
    """Classe auxiliar para criar e gerenciar estrutura de diretórios."""
    
    ESTRUTURA_DIRETORIOS = {
        'resultados_2/parte1/csv': [],
        'resultados_2/parte1/grafos': [],
        'resultados_2/parte1/graficos': [],
        'resultados_2/parte2/csv': [],
        'resultados_2/parte2/grafos': [],
        'resultados_2/parte2/graficos': [],
    }
    
    @staticmethod
    def criar_estrutura():
        """Cria a estrutura de diretórios completa."""
        for diretorio in ConfiguradorDiretorios.ESTRUTURA_DIRETORIOS.keys():
            Path(diretorio).mkdir(parents=True, exist_ok=True)
        print(f"✓ Estrutura de diretórios criada com sucesso")
    
    @staticmethod
    def obter_caminho_arquivo(tipo_parte: str, tipo_saida: str, nome_arquivo: str) -> str:
        """
        Retorna caminho completo para arquivo baseado em tipo.
        
        Args:
            tipo_parte: 'parte1' ou 'parte2'
            tipo_saida: 'csv', 'grafos' ou 'graficos'
            nome_arquivo: nome do arquivo
        
        Returns:
            Caminho completo do arquivo
        """
        return f"resultados_2/{tipo_parte}/{tipo_saida}/{nome_arquivo}"

# Criar estrutura de diretórios
ConfiguradorDiretorios.criar_estrutura()

✓ Estrutura de diretórios criada com sucesso


# 1. Configuração Inicial e Estrutura de Diretórios

## Objetivo
Criar e gerenciar a estrutura de diretórios para organizar todos os resultados em `resultados_2/`.

## Estrutura de Diretórios Criada

```
resultados_2/
├── parte1/                 (Força Bruta - Instâncias Aleatórias)
│   ├── csv/                # parametros_grafos.csv, resultados_forca_bruta.csv
│   ├── grafos/             # Visualizações dos grafos coloridos (PNG)
│   └── graficos/           # Gráficos de análise e escalabilidade (PNG)
│
└── parte2/                 (Welsh-Powell - Instâncias DIMACS)
    ├── csv/                # informacoes_instancias.csv, resultados_welsh_powell.csv
    ├── grafos/             # Visualizações dos grafos coloridos (PNG)
    └── graficos/           # Gráficos comparativos (PNG)
```

## Classe ConfiguradorDiretorios

Implementa a criação automática e gerenciamento de caminhos:

### Métodos Principais
- **`criar_estrutura()`**: Cria todos os diretórios necessários
- **`obter_caminho_arquivo(tipo_parte, tipo_saida, nome_arquivo)`**: Retorna caminho completo

### Exemplo de Uso
```python
# Criar estrutura
ConfiguradorDiretorios.criar_estrutura()

# Obter caminho para salvar CSV
caminho = ConfiguradorDiretorios.obter_caminho_arquivo(
    'parte1', 'csv', 'parametros_grafos.csv'
)
# Retorna: 'resultados_2/parte1/csv/parametros_grafos.csv'
```

## Vantagens
✓ Organização clara e consistente
✓ Fácil localizar resultados posteriores
✓ Suporta múltiplas partes/análises
✓ Automático - sem criar diretórios manualmente


In [3]:
def extrair_parametros_grafo(grafo: nx.Graph) -> Dict:
    """
    Extrai parâmetros estatísticos de um grafo.
    
    Função auxiliar comum usada em ambas as partes para extrair informações
    de um grafo (densidade, grau médio, conectividade, etc.).
    
    Args:
        grafo: Grafo NetworkX
        
    Returns:
        Dicionário com parâmetros do grafo
    """
    return {
        'num_vertices': grafo.number_of_nodes(),
        'num_arestas': grafo.number_of_edges(),
        'densidade': nx.density(grafo),
        'grau_medio': sum(dict(grafo.degree()).values()) / grafo.number_of_nodes(),
        'grau_minimo': min(dict(grafo.degree()).values()),
        'grau_maximo': max(dict(grafo.degree()).values()),
        'eh_conexo': nx.is_connected(grafo),
        'num_componentes': nx.number_connected_components(grafo),
    }

print("✓ Função extrair_parametros_grafo definida")

✓ Função extrair_parametros_grafo definida


In [4]:
def ler_arquivo_dimacs(caminho_arquivo: str) -> nx.Graph:
    """
    Função auxiliar comum: Lê arquivo DIMACS e cria grafo NetworkX.
    
    Formato DIMACS:
    - Linhas começadas com 'c' são comentários
    - Linha 'p edge <n> <m>' define n vértices e m arestas
    - Linhas 'e <u> <v>' definem arestas
    
    Args:
        caminho_arquivo: Caminho do arquivo DIMACS
        
    Returns:
        Grafo NetworkX ou None se arquivo não existir
    """
    if not os.path.exists(caminho_arquivo):
        print(f"⚠ Arquivo não encontrado: {caminho_arquivo}")
        return None
    
    grafo = nx.Graph()
    
    try:
        with open(caminho_arquivo, 'r', encoding='utf-8', errors='ignore') as f:
            for linha in f:
                linha = linha.strip()
                
                # Ignorar comentários
                if linha.startswith('c'):
                    continue
                
                # Linha de problema (define número de vértices)
                if linha.startswith('p edge'):
                    partes = linha.split()
                    num_vertices = int(partes[2])
                    num_arestas = int(partes[3])
                    grafo.add_nodes_from(range(1, num_vertices + 1))
                    continue
                
                # Linhas de aresta
                if linha.startswith('e'):
                    partes = linha.split()
                    u = int(partes[1])
                    v = int(partes[2])
                    grafo.add_edge(u, v)
        
        return grafo
    
    except Exception as e:
        print(f"⚠ Erro ao ler arquivo DIMACS: {str(e)}")
        return None

print("✓ Função ler_arquivo_dimacs definida")

✓ Função ler_arquivo_dimacs definida


In [5]:
def visualizar_e_salvar_grafo(grafo: nx.Graph, coloracao: Dict, caminho_saida: str):
    """
    Visualiza grafo com coloração e salva como PNG.
    
    Função auxiliar comum que cria visualização dos grafos coloridos em ambas
    as partes da solução. Adapta a visualização conforme o tamanho do grafo.
    Implementa visualização adaptativa com estatísticas visuais.
    
    Args:
        grafo: Grafo NetworkX
        coloracao: Dicionário {vertice: cor}
        caminho_saida: Caminho para salvar arquivo PNG
    """
    num_vertices = grafo.number_of_nodes()
    num_arestas = grafo.number_of_edges()
    cores_set = set(coloracao.values())
    num_cores = max(cores_set) + 1
    cores_lista = [coloracao[node] for node in grafo.nodes()]
    densidade = nx.density(grafo)
    
    # Calcular distribuição de cores
    distribuicao_cores = {}
    for cor in cores_set:
        distribuicao_cores[cor] = sum(1 for v in coloracao.values() if v == cor)
    
    # Criar figura com subplot para grafos grandes (inclui estatísticas)
    if num_vertices <= 100:
        # ======= GRAFOS MUITO PEQUENOS (≤100): Visualização Completa com Labels =======
        fig = plt.figure(figsize=(12, 8))
        
        # Subplot principal: grafo
        ax_grafo = plt.subplot(1, 2, 1)
        pos = nx.spring_layout(grafo, seed=42, iterations=50)
        
        nx.draw_networkx_edges(grafo, pos, alpha=0.3, width=0.8, ax=ax_grafo)
        nx.draw_networkx_nodes(grafo, pos, node_color=cores_lista, 
                              node_size=500, cmap='Set3', vmin=0, 
                              vmax=max(cores_set), ax=ax_grafo)
        nx.draw_networkx_labels(grafo, pos, font_size=7, ax=ax_grafo)
        
        ax_grafo.set_title(f'Coloração de Vértices\n{num_vertices} vértices, {num_arestas} arestas',
                          fontsize=12, fontweight='bold')
        ax_grafo.axis('off')
        
        # Subplot de estatísticas: distribuição de cores
        ax_stats = plt.subplot(1, 2, 2)
        cores_ids = sorted(distribuicao_cores.keys())
        contagens = [distribuicao_cores[c] for c in cores_ids]
        colors_bar = plt.cm.Set3(np.linspace(0, 1, num_cores))
        
        bars = ax_stats.bar(cores_ids, contagens, color=colors_bar[:len(cores_ids)], 
                            edgecolor='black', linewidth=1.5)
        ax_stats.set_xlabel('Cor', fontsize=11, fontweight='bold')
        ax_stats.set_ylabel('Número de Vértices', fontsize=11, fontweight='bold')
        ax_stats.set_title('Distribuição de Cores', fontsize=12, fontweight='bold')
        ax_stats.grid(True, alpha=0.3, axis='y')
        
        # Adicionar valores nas barras
        for bar, count in zip(bars, contagens):
            height = bar.get_height()
            ax_stats.text(bar.get_x() + bar.get_width()/2., height,
                         f'{int(count)}', ha='center', va='bottom', fontweight='bold')
        
        # Adicionar informações estatísticas
        info_text = f"χ(G) = {num_cores} cores\nDensidade: {densidade:.4f}\nGrau médio: {2*num_arestas/num_vertices:.2f}"
        ax_stats.text(0.02, 0.98, info_text, transform=ax_stats.transAxes,
                     fontsize=10, verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        dpi = 150
        
    elif num_vertices <= 1000:
        # ======= GRAFOS PEQUENOS/MÉDIOS (101-1000): Layout Otimizado + Stats =======
        # Inclui instâncias a (450) e b (864) e c (1000)
        fig = plt.figure(figsize=(14, 6))
        
        # Subplot principal: grafo
        ax_grafo = plt.subplot(1, 3, (1, 2))
        try:
            pos = nx.kamada_kawai_layout(grafo)
        except:
            pos = nx.spring_layout(grafo, seed=42, iterations=30, k=0.15)
        
        nx.draw_networkx_edges(grafo, pos, alpha=0.15, width=0.3, ax=ax_grafo)
        nx.draw_networkx_nodes(grafo, pos, node_color=cores_lista,
                              node_size=150, cmap='Set3', vmin=0,
                              vmax=max(cores_set), ax=ax_grafo, edgecolors='black', linewidths=0.5)
        
        ax_grafo.set_title(f'Coloração de Vértices - {num_vertices} vértices\nχ(G) = {num_cores} cores',
                          fontsize=12, fontweight='bold')
        ax_grafo.axis('off')
        
        # Subplot de estatísticas
        ax_stats = plt.subplot(1, 3, 3)
        cores_ids = sorted(distribuicao_cores.keys())
        contagens = [distribuicao_cores[c] for c in cores_ids]
        colors_bar = plt.cm.Set3(np.linspace(0, 1, num_cores))
        
        ax_stats.barh(cores_ids, contagens, color=colors_bar[:len(cores_ids)],
                     edgecolor='black', linewidth=1)
        ax_stats.set_ylabel('Cor', fontsize=10, fontweight='bold')
        ax_stats.set_xlabel('Vértices', fontsize=10, fontweight='bold')
        ax_stats.set_title('Distribuição', fontsize=11, fontweight='bold')
        ax_stats.grid(True, alpha=0.3, axis='x')
        ax_stats.invert_yaxis()
        
        info_text = f"Vértices: {num_vertices}\nArestas: {num_arestas}\nχ(G): {num_cores}\nDensidade: {densidade:.5f}\nGrau médio: {2*num_arestas/num_vertices:.2f}"
        ax_stats.text(0.02, 0.02, info_text, transform=ax_stats.transAxes,
                     fontsize=9, verticalalignment='bottom',
                     bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        
        dpi = 120
        
    else:
        # ======= GRAFOS GRANDES (>1000): Visualização Simplificada Consistente =======
        fig = plt.figure(figsize=(14, 6))
        
        # Subplot 1: Grafo simplificado (ocupa 2/3 do espaço)
        ax_grafo = plt.subplot(1, 3, (1, 2))
        try:
            # Para grafos muito grandes, usar layout mais rápido
            pos = nx.spring_layout(grafo, seed=42, iterations=15, k=0.2)
        except:
            pos = nx.random_layout(grafo, seed=42)
        
        nx.draw_networkx_edges(grafo, pos, alpha=0.08, width=0.2, ax=ax_grafo)
        nx.draw_networkx_nodes(grafo, pos, node_color=cores_lista,
                              node_size=50, cmap='Set3', vmin=0,
                              vmax=max(cores_set), ax=ax_grafo, linewidths=0)
        
        ax_grafo.set_title(f'Coloração de Vértices - {num_vertices} vértices\nχ(G) = {num_cores} cores',
                          fontsize=12, fontweight='bold')
        ax_grafo.axis('off')
        
        # Subplot 2: Distribuição de cores (barras horizontais + info)
        ax_stats = plt.subplot(1, 3, 3)
        cores_ids = sorted(distribuicao_cores.keys())
        contagens = [distribuicao_cores[c] for c in cores_ids]
        colors_bar = plt.cm.Set3(np.linspace(0, 1, num_cores))
        
        ax_stats.barh(cores_ids, contagens, color=colors_bar[:len(cores_ids)],
                     edgecolor='black', linewidth=1)
        ax_stats.set_ylabel('Cor', fontsize=10, fontweight='bold')
        ax_stats.set_xlabel('Vértices', fontsize=10, fontweight='bold')
        ax_stats.set_title('Distribuição', fontsize=11, fontweight='bold')
        ax_stats.grid(True, alpha=0.3, axis='x')
        ax_stats.invert_yaxis()
        
        info_text = f"Vértices: {num_vertices}\nArestas: {num_arestas}\nχ(G): {num_cores}\nDensidade: {densidade:.5f}\nGrau médio: {2*num_arestas/num_vertices:.2f}"
        ax_stats.text(0.02, 0.02, info_text, transform=ax_stats.transAxes,
                     fontsize=9, verticalalignment='bottom',
                     bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        
        dpi = 120
    
    plt.tight_layout()
    plt.savefig(caminho_saida, dpi=dpi, bbox_inches='tight')
    plt.close()

print("✓ Função visualizar_e_salvar_grafo definida")

✓ Função visualizar_e_salvar_grafo definida


# Estrutura da Solução Completa

Este notebook implementa uma **solução completa** para o Problema da Coloração de Grafos em duas etapas:

## 📋 Organização (2 Algoritmos)

| Parte | Algoritmo | Tipo de Instância | Quantidade | Objetivo |
|-------|-----------|-------------------|-----------|----------|
| **Parte 1** | Força Bruta | Aleatória (5-13v) | 27 grafos | χ(G) exato |
| **Parte 2** | Welsh-Powell | DIMACS (450-4730v) | 5 grafos | Aproximação rápida |

**Total**: **2 algoritmos**, **32 instâncias** processadas

## 📊 Fluxo Geral

```
1. Gerar/Carregar Instâncias
        ↓
2. Executar Algoritmos
        ↓
3. Medir Tempo e Validar
        ↓
4. Salvar Resultados em CSV
        ↓
5. Gerar Gráficos e Visualizações
        ↓
6. Consolidar Dados
```

## 📁 Diretórios

Todos os resultados são organizados em `resultados_2/` com estrutura hierárquica:
- `parte1/csv/`: Parâmetros e resultados força bruta
- `parte1/grafos/`: Visualizações dos grafos
- `parte1/graficos/`: Gráficos de análise
- `parte2/csv/`: Resultados Welsh-Powell
- `parte2/grafos/`: Visualizações DIMACS

- `parte2/graficos/`: Gráficos comparativos

## Análises Implementadas

### 1️⃣ Força Bruta (Parte 1)
- **Algoritmo**: Teste sistemático de k-colorações
- **Instâncias**: 27 grafos aleatórios (5-13 vértices)
- **Garantia**: χ(G) exato
- **Arquivo de saída**: `resultados_forca_bruta.csv`

### 2️⃣ Heurística Welsh-Powell (Parte 2)
- **Algoritmo**: Greedy com ordenação por grau
- **Instâncias**: 5 grafos DIMACS (le450, inithx, r1000, ash958, wap03a)
- **Qualidade**: Aproximação do ótimo
- **Arquivo de saída**: `resultados_welsh_powell.csv`

## Estrutura de Saída
Todos os resultados salvos em `resultados_2/` com organização hierárquica:
- `parte1/`: Força Bruta
- `parte2/`: Welsh-Powell
- Cada parte contém: `csv/`, `grafos/`, `graficos/`

In [6]:
def gerar_instancia_grafo_aleatorio(num_vertices: int, probabilidade: float, 
                                   seed: int = None) -> nx.Graph:
    """
    REQUISITO 2 - PARTE 1: Gera instância aleatória de grafo.
    
    Cria um grafo aleatório usando modelo Erdős-Rényi:
    - Não direcionado
    - Não ponderado
    - Sem loops ou arestas paralelas (garantido pelo modelo)
    
    Args:
        num_vertices: Número de vértices (n)
        probabilidade: Probabilidade de aresta entre dois vértices (0 a 1)
        seed: Seed para reprodutibilidade
    
    Returns:
        Grafo NetworkX não direcionado
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    
    # Modelo Erdős-Rényi: grafo aleatório
    grafo = nx.erdos_renyi_graph(n=num_vertices, p=probabilidade)
    
    # Garantir que o grafo tem arestas (para grafos muito pequenos)
    while grafo.number_of_edges() == 0 and num_vertices > 1:
        grafo = nx.erdos_renyi_graph(n=num_vertices, p=probabilidade)
    
    return grafo

print("✓ Função gerar_instancia_grafo_aleatorio definida")

✓ Função gerar_instancia_grafo_aleatorio definida


# Parte 1: Força Bruta em Instâncias Aleatórias

## Objetivo
Executar força bruta em 27 grafos gerados aleatoriamente para determinar seu número cromático exato.

## Instâncias
- **Tamanhos**: 5 até 13 vértices (9 tamanhos)
- **Quantidade**: 3 grafos por tamanho (27 total)
- **Geração**: Aleatória com densidade proporcional

## Algoritmo
**Força Bruta** - Teste sistemático de k-colorações (k = 1, 2, 3, ...)

### Estratégia:
1. Incrementa k começando em 1
2. Tenta colorir o grafo com k cores
3. Se bem-sucedido, retorna k (número cromático exato)
4. Se falhar, incrementa k e tenta novamente

### Garantia:
- Encontra sempre o número cromático χ(G)
- Custo computacional elevado para grafos grandes
- Adequado para instâncias até ~13 vértices

## Arquivos de Saída
- CSVs com resultados em `resultados_2/parte1/csv/`
- Gráficos em `resultados_2/parte1/graficos/`
- Visualizações em `resultados_2/parte1/grafos/`


In [7]:
def algoritmo_forca_bruta_coloracao(grafo: nx.Graph) -> Tuple[int, Dict, float]:
    """
    REQUISITO 1 - PARTE 1: Implementa algoritmo de força bruta para coloração.
    
    ESTRATÉGIA OTIMIZADA COM BACKTRACKING E PODA:
    -----------------------------------------------
    Para cada número de cores k = 1, 2, 3, ...:
      1. Usa backtracking para colorir vértices um por um
      2. Descarta (poda) caminhos inválidos imediatamente
      3. Se encontrar coloração válida, retorna k como número cromático
    
    OTIMIZAÇÕES:
    - Backtracking: Para imediatamente ao detectar conflito
    - Poda: Não explora ramos inválidos (reduz k^n drasticamente)
    - Ordenação: Processa vértices por grau decrescente (melhor poda)
    
    Complexidade: O(k^n) pior caso, mas com poda eficiente na prática
    Viável para n ≤ ~15-20 vértices
    
    Args:
        grafo: Grafo NetworkX
        
    Returns:
        Tupla: (número_cromático, coloração_dict, tempo_segundos)
    """
    inicio = time.time()
    
    vertices = list(grafo.nodes())
    n_vertices = len(vertices)
    
    # Ordenar vértices por grau decrescente para melhor poda
    vertices_ordenados = sorted(vertices, key=lambda v: grafo.degree(v), reverse=True)
    
    # Pré-calcular vizinhos para acesso rápido
    vizinhos = {v: set(grafo.neighbors(v)) for v in vertices}
    
    def backtrack(indice: int, coloracao: Dict, num_cores: int) -> bool:
        """
        Tenta colorir vértices a partir do índice usando backtracking.
        
        Args:
            indice: Índice do vértice atual
            coloracao: Dicionário de coloração atual
            num_cores: Número máximo de cores disponíveis
            
        Returns:
            True se encontrou coloração válida, False caso contrário
        """
        # Caso base: todos os vértices foram coloridos
        if indice == n_vertices:
            return True
        
        vertice_atual = vertices_ordenados[indice]
        
        # Determinar cores já usadas por vizinhos
        cores_vizinhos = set()
        for vizinho in vizinhos[vertice_atual]:
            if vizinho in coloracao:
                cores_vizinhos.add(coloracao[vizinho])
        
        # Tentar cada cor disponível
        for cor in range(num_cores):
            # PODA: Se cor já usada por vizinho, pular
            if cor in cores_vizinhos:
                continue
            
            # Atribuir cor ao vértice atual
            coloracao[vertice_atual] = cor
            
            # Recursivamente colorir próximo vértice
            if backtrack(indice + 1, coloracao, num_cores):
                return True
            
            # Backtrack: remover coloração
            del coloracao[vertice_atual]
        
        # Nenhuma cor funcionou
        return False
    
    # Tentar com 1, 2, 3, ... cores até encontrar coloração válida
    for num_cores in range(1, n_vertices + 1):
        coloracao = {}
        if backtrack(0, coloracao, num_cores):
            tempo = time.time() - inicio
            return num_cores, coloracao, tempo
    
    # Nunca deve chegar aqui (coloração sempre existe)
    tempo = time.time() - inicio
    return n_vertices, {v: i for i, v in enumerate(vertices)}, tempo

print("✓ Função algoritmo_forca_bruta_coloracao definida (OTIMIZADA com backtracking e poda)")

✓ Função algoritmo_forca_bruta_coloracao definida (OTIMIZADA com backtracking e poda)


# Parte 2: Heurística Welsh-Powell em Instâncias DIMACS

## Objetivo
Aplicar a heurística Welsh-Powell em 5 instâncias DIMACS do repositório de testes padrão.

## Instâncias DIMACS
5 grafos de teste com estruturas variadas:
- **le450_25a.col**: Grafo denso, ~450 vértices
- **inithx.i.1.col**: Grafo médio
- **r1000.1.col**: Grafo aleatório, ~1000 vértices
- **ash958GPIA.col**: Grafo denso, ~958 vértices
- **wap03a.col**: Grafo de aplicação prática

## Algoritmo
**Welsh-Powell** - Heurística greedy com ordenação por grau

### Estratégia:
1. Ordena vértices por grau decrescente
2. Atribui cores greedily (menor disponível)
3. Rápido e com qualidade razoável

### Resultados Esperados:
- Cores encontradas para cada instância
- Tempo de execução
- Comparação com limites teóricos

## Arquivos de Saída
- CSVs com resultados em `resultados_2/parte2/csv/`
- Gráficos em `resultados_2/parte2/graficos/`
- Visualizações em `resultados_2/parte2/grafos/`


In [8]:
def processar_instancias_por_tamanho(tamanho_min: int = 5, tamanho_max: int = 13,
                                     probabilidade: float = 0.3,
                                     num_instancias_por_tamanho: int = 3) -> Tuple[List, List]:
    """
    REQUISITO 2 + 3 - PARTE 1: Processa múltiplas instâncias aleatórias.
    
    Gera instâncias de tamanho tamanho_min a tamanho_max e aplica força bruta em cada.
    Mede tempo de execução para cada instância.
    
    Args:
        tamanho_min: Tamanho mínimo (padrão 5 vértices)
        tamanho_max: Tamanho máximo (padrão 13 vértices)
        probabilidade: Probabilidade de aresta entre vértices
        num_instancias_por_tamanho: Número de instâncias por tamanho
        
    Returns:
        Tupla: (lista_parametros, lista_resultados)
    """
    lista_parametros = []
    lista_resultados = []
    
    print("\n" + "="*70)
    print("PARTE 1: PROCESSANDO INSTÂNCIAS ALEATÓRIAS COM FORÇA BRUTA")
    print("="*70)
    
    for tamanho in range(tamanho_min, tamanho_max + 1):
        print(f"\nTamanho: {tamanho} vértices", end="")
        
        for instancia_idx in range(num_instancias_por_tamanho):
            # Gerar instância
            seed = tamanho * 1000 + instancia_idx
            grafo = gerar_instancia_grafo_aleatorio(tamanho, probabilidade, seed=seed)
            
            # Extrair parâmetros
            parametros = extrair_parametros_grafo(grafo)
            parametros['tamanho'] = tamanho
            parametros['instancia'] = instancia_idx + 1
            parametros['id_grafo'] = f"n{tamanho:02d}_i{instancia_idx+1:02d}"
            
            # Aplicar força bruta
            try:
                num_cromatico, coloracao, tempo = algoritmo_forca_bruta_coloracao(grafo)
                
                resultado = {
                    'id_grafo': parametros['id_grafo'],
                    'tamanho': tamanho,
                    'instancia': instancia_idx + 1,
                    'numero_cromatico': num_cromatico,
                    'tempo_segundos': tempo,
                    'arestas': parametros['num_arestas'],
                }
                
                lista_parametros.append(parametros)
                lista_resultados.append(resultado)
                
                # Salvar visualização
                caminho_grafo = ConfiguradorDiretorios.obter_caminho_arquivo(
                    'parte1', 'grafos', f"grafo_{parametros['id_grafo']}.png"
                )
                visualizar_e_salvar_grafo(grafo, coloracao, caminho_grafo)
                
                print(".", end="", flush=True)
            
            except KeyboardInterrupt:
                print("\n⚠ Execução interrompida pelo usuário")
                break
            except Exception as e:
                print(f"\n⚠ Erro ao processar grafo {tamanho}_{instancia_idx}: {str(e)}")
                continue
        
        print(f" [{tamanho} vértices processado]")
    
    print("\n✓ Processamento de instâncias concluído")
    return lista_parametros, lista_resultados

print("✓ Função processar_instancias_por_tamanho definida")

✓ Função processar_instancias_por_tamanho definida


# 2. Funções Auxiliares para Processamento

## Objetivo desta Seção
Implementar funções auxiliares para manipulação de grafos e processamento de dados DIMACS.

## Funções Principais

### `carregar_grafo_dimacs(filepath: str)`
Carrega um grafo a partir de um arquivo DIMACS (.col).

**Formato DIMACS:**
```
c comentário
p edge V E
1 2    (aresta entre vértices 1 e 2)
```

**Retorno:**
- NetworkX Graph com V vértices e E arestas

### `eh_coloracao_valida(grafo, coloracao)`
Verifica se uma coloração é válida (sem vértices adjacentes com a mesma cor).

**Validação:**
- Para cada aresta (u, v): coloracao[u] ≠ coloracao[v]
- Retorna `True` se válida, `False` caso contrário

### `obter_densidade_grafo(grafo: nx.Graph)`
Calcula a densidade do grafo: ρ(G) = 2|E| / (|V|(|V|-1))

**Interpretação:**
- ρ = 0: Sem arestas (vazio)
- ρ = 1: Completo
- 0 < ρ < 1: Densidade intermediária

## Tipos de Dados Utilizados
- `grafo`: Objeto NetworkX Graph
- `coloracao`: Dict {vértice: cor_inteira}
- `tempo`: Float (segundos)


# 3. Definição da Força Bruta

## Objetivo desta Seção
Implementar o algoritmo de força bruta para encontrar o número cromático exato χ(G).

## Algoritmo Força Bruta

### Conceito Principal
Testa sistematicamente todas as k-colorações possíveis começando com k=1, até encontrar uma coloração válida.

### Passos do Algoritmo
1. **Incremento de k**: Começa com k=1
2. **Teste**: Tenta colorir o grafo com k cores
3. **Verificação**: Se bem-sucedido, retorna k (número cromático)
4. **Repetição**: Se falhar, incrementa k e tenta novamente

### Complexidade
- **Tempo**: O(k³) onde k é o número cromático encontrado
- **Espaço**: O(n) para manter a coloração

### Propriedades
- ✓ Sempre encontra a solução ótima (número cromático exato)
- ✗ Complexidade exponencial em relação ao tamanho do grafo
- ✓ Adequado para grafos pequenos (até ~13 vértices)

### Implementação
- Gera configurações de coloração aleatoriamente
- Verifica se a coloração é válida
- Mantém resultado com k mínimo


In [9]:
def algoritmo_welsh_powell(grafo: nx.Graph) -> Tuple[int, Dict, float]:
    """
    REQUISITO 1 - PARTE 2: Implementa heurística Welsh-Powell para coloração.
    
    ESTRATÉGIA:
    -----------
    Algoritmo guloso que ordena vértices por grau decrescente:
    1. Ordena vértices por grau (maior primeiro)
    2. Para cada vértice em ordem:
       - Atribui a menor cor disponível (que não foi usada por vizinhos)
    
    Complexidade: O(n + m) - Linear em número de vértices e arestas
    Garantias: Encontra coloração válida (nem sempre ótima)
    Qualidade: Típicamente O(Δ + 1) cores, onde Δ é grau máximo
    
    Args:
        grafo: Grafo NetworkX
        
    Returns:
        Tupla: (número_cores_usado, coloracao_dict, tempo_segundos)
    """
    inicio = time.time()
    
    # Ordenar vértices por grau decrescente (heurística Welsh-Powell)
    graus = dict(grafo.degree())
    vertices_ordenados = sorted(graus.keys(), key=lambda v: graus[v], reverse=True)
    
    coloracao = {}
    
    # Para cada vértice em ordem de grau decrescente
    for vertice in vertices_ordenados:
        # Cores já usadas pelos vizinhos
        cores_vizinhos = set()
        for vizinho in grafo.neighbors(vertice):
            if vizinho in coloracao:
                cores_vizinhos.add(coloracao[vizinho])
        
        # Encontrar menor cor disponível
        cor = 0
        while cor in cores_vizinhos:
            cor += 1
        
        coloracao[vertice] = cor
    
    tempo = time.time() - inicio
    num_cores = max(coloracao.values()) + 1 if coloracao else 0
    
    return num_cores, coloracao, tempo

print("✓ Função algoritmo_welsh_powell definida")

✓ Função algoritmo_welsh_powell definida


# 4. Definição da Heurística Welsh-Powell

## Objetivo desta Seção
Implementar a heurística Welsh-Powell para coloração de vértices, um algoritmo greedy eficiente baseado na ordenação por grau.

## Algoritmo Welsh-Powell

### Conceito Principal
Ordena os vértices em ordem decrescente de grau e aplica uma estratégia greedy para atribuir cores.

### Passos do Algoritmo
1. **Ordenação**: Ordena todos os vértices por grau decrescente
2. **Inicialização**: Cria lista de cores disponíveis
3. **Atribuição**: Para cada vértice na ordem:
   - Atribui a cor mínima disponível (greedy)
   - Uma cor é "disponível" se nenhum vizinho a usa
4. **Retorno**: Número total de cores e mapeamento vértice → cor

### Complexidade
- **Tempo**: O(n log n) para ordenação + O(n²) para atribuição = O(n²)
- **Espaço**: O(n) para estruturas de cores

### Propriedades
- ✓ Sempre termina com uma coloração válida
- ✓ Qualidade: Frequentemente ≤ Δ(G) + 1 (onde Δ é o grau máximo)
- ✗ Não garante optimalidade

### Vantagens para este Projeto
- Rápido para instâncias DIMACS
- Boa qualidade de solução
- Determinístico
- Simples de implementar e verificar


# Algoritmos da Solução

## 🔴 Força Bruta (O(k³))
**Características:**
- Estratégia: Teste de todas as k-colorações possíveis (k = 1, 2, 3, ...)
- Parada: Encontra o primeiro k que funciona (número cromático exato)
- Complexidade: Exponencial em relação ao número de vértices
- Garantia: Sempre encontra a solução ótima
- Aplicação: Instâncias pequenas/médias (até ~15 vértices)

**Implementação:**
- Gera coloração aleatória inicial
- Incrementa k e tenta novamente se falhar
- Retorna χ(G) exato e mapeamento de cores

## 🟢 Heurística Welsh-Powell (O(n log n))
**Características:**
- Estratégia: Greedy com ordenação por grau decrescente
- Ordem: Vértice com maior grau recebe cor primeiro
- Complexidade: Polinomial (rápida)
- Qualidade: Aproximação razoável do ótimo
- Aplicação: Instâncias médias/grandes

**Implementação:**
- Ordena vértices por grau decrescente
- Atribui menor cor disponível (greedy)
- Retorna número de cores e mapeamento

## 📊 Comparação
| Aspecto | Força Bruta | Welsh-Powell |
|---------|-------------|--------------|
| **Optimalidade** | Exato | Aproximado |
| **Velocidade** | Lenta | Rápida |
| **Instâncias** | Pequenas | Grandes |
| **Overhead** | Computacional alto | Baixo |


In [10]:
def processar_instancias_dimacs() -> List:
    """
    REQUISITO 2 + 3 - PARTE 2: Processa instâncias DIMACS com heurística.
    
    Nota: Este é um template. Você deve colocar os arquivos DIMACS em
    'instancias/' para processar as 5 instâncias do Moodle.
    
    Instâncias esperadas (do Moodle):
      a. 450 vértices, 8260 arestas
      b. 864 vértices, 18707 arestas
      c. 1000 vértices, 14378 arestas
      d. 1916 vértices, 12506 arestas
      e. 4730 vértices, 286722 arestas
    
    Returns:
        Lista de resultados para cada instância
    """
    lista_resultados = []
    
    # Definir instâncias esperadas (atualizar conforme necessário)
    instancias = [
        {
            'id': 'a',
            'caminho': 'instancias/a - le450_25a.col.txt',
            'vertices_esperados': 450,
            'arestas_esperadas': 8260,
        },
        {
            'id': 'b',
            'caminho': 'instancias/b - inithx.i.1.col.txt',
            'vertices_esperados': 864,
            'arestas_esperadas': 18707,
        },
        {
            'id': 'c',
            'caminho': 'instancias/c - r1000.1.col.txt',
            'vertices_esperados': 1000,
            'arestas_esperadas': 14378,
        },
        {
            'id': 'd',
            'caminho': 'instancias/d - ash958GPIA.col.txt',
            'vertices_esperados': 1916,
            'arestas_esperadas': 12506,
        },
        {
            'id': 'e',
            'caminho': 'instancias/e - wap03a.col.txt',
            'vertices_esperados': 4730,
            'arestas_esperadas': 286722,
        },
    ]
    
    print("\n" + "="*70)
    print("PARTE 2: PROCESSANDO INSTÂNCIAS DIMACS COM HEURÍSTICA")
    print("="*70)
    
    for instancia in instancias:
        print(f"\nProcessando instância {instancia['id']}... ", end="", flush=True)
        
        grafo = ler_arquivo_dimacs(instancia['caminho'])
        
        if grafo is None:
            print(f"⚠ Não encontrado")
            continue
        
        # Verificar se grafo tem vértices
        if grafo.number_of_nodes() == 0:
            print(f"⚠ Grafo vazio")
            continue
        
        try:
            # Aplicar heurística Welsh-Powell
            num_cores, coloracao, tempo = algoritmo_welsh_powell(grafo)
            
            resultado = {
                'instancia_id': instancia['id'],
                'caminho': instancia['caminho'],
                'num_vertices': grafo.number_of_nodes(),
                'num_arestas': grafo.number_of_edges(),
                'cores_heuristica': num_cores,
                'tempo_segundos': tempo,
                'densidade': nx.density(grafo),
                'grau_medio': sum(dict(grafo.degree()).values()) / grafo.number_of_nodes(),
                'coloracao': coloracao,
                'grafo': grafo
            }
            
            lista_resultados.append(resultado)
            
            # Salvar visualização do grafo colorido
            caminho_grafo = ConfiguradorDiretorios.obter_caminho_arquivo(
                'parte2', 'grafos', f'instancia_{instancia["id"]}.png'
            )
            visualizar_e_salvar_grafo(grafo, coloracao, caminho_grafo)
            
            print(f"✓ {num_cores} cores em {tempo:.4f}s")
        
        except Exception as e:
            print(f"⚠ Erro: {str(e)}")
            continue
    
    print("\n✓ Processamento de instâncias DIMACS concluído")
    return lista_resultados

print("✓ Função processar_instancias_dimacs definida")

✓ Função processar_instancias_dimacs definida


# Execução da Parte 1: Força Bruta em Instâncias Aleatórias

## Objetivo
Executar força bruta em 27 grafos aleatórios gerados proceduralmente.

## Instâncias
- 9 tamanhos diferentes: 5, 6, 7, 8, 9, 10, 11, 12, 13 vértices
- 3 grafos por tamanho (27 total)
- Gerados aleatoriamente com arestas proporcionais ao tamanho

## Algoritmo Força Bruta
- **Estratégia**: Teste de todas as colorações possíveis até encontrar a mínima
- **Complexidade**: Exponencial (k³)
- **Garantia**: Encontra o número cromático exato
- **Saída**: Número de cores mínimas e sequência de coloração

## Arquivos de Saída
- `resultados_2/parte1/csv/parametros_grafos.csv` - Parâmetros estruturais (v, e, densidade, grau médio)
- `resultados_2/parte1/csv/resultados_forca_bruta.csv` - Número cromático encontrado para cada instância
- Gráficos em `resultados_2/parte1/graficos/` e visualizações em `resultados_2/parte1/grafos/`


In [11]:
def salvar_resultados_csv_parte1(parametros_list: List, resultados_list: List):
    """
    Salva resultados da Parte 1 em CSV.
    
    Args:
        parametros_list: Lista de dicionários com parâmetros dos grafos
        resultados_list: Lista de dicionários com resultados da força bruta
    """
    # CSV de parâmetros
    if parametros_list:
        df_parametros = pd.DataFrame(parametros_list)
        caminho = ConfiguradorDiretorios.obter_caminho_arquivo(
            'parte1', 'csv', 'parametros_grafos.csv'
        )
        df_parametros.to_csv(caminho, index=False)
        print(f"✓ Parâmetros salvos em: {caminho}")
    
    # CSV de resultados
    if resultados_list:
        df_resultados = pd.DataFrame(resultados_list)
        caminho = ConfiguradorDiretorios.obter_caminho_arquivo(
            'parte1', 'csv', 'resultados_forca_bruta.csv'
        )
        df_resultados.to_csv(caminho, index=False)
        print(f"✓ Resultados salvos em: {caminho}")

def salvar_resultados_csv_parte2(resultados_list: List):
    """
    Salva resultados da Parte 2 em CSV.
    
    Args:
        resultados_list: Lista de dicionários com resultados da heurística
    """
    if resultados_list:
        df_resultados = pd.DataFrame(resultados_list)
        caminho = ConfiguradorDiretorios.obter_caminho_arquivo(
            'parte2', 'csv', 'resultados_heuristica.csv'
        )
        df_resultados.to_csv(caminho, index=False)
        print(f"✓ Resultados heurística salvos em: {caminho}")

print("✓ Funções de exportação CSV definidas")

✓ Funções de exportação CSV definidas


In [12]:
def salvar_informacoes_instancias(resultados_list: List):
    """
    Salva informações estruturais das instâncias DIMACS em CSV.
    
    Gera um CSV com metadados de cada instância:
    - Nome da instância
    - Número de vértices
    - Número de arestas
    - Densidade do grafo
    - Grau médio
    - Número de cores encontradas
    - Tempo de execução
    
    Args:
        resultados_list: Lista de dicionários com resultados da heurística
    """
    if not resultados_list:
        print("⚠ Sem dados para gerar CSV de informações das instâncias")
        return
    
    # Extrair e calcular informações relevantes
    info_instancias = []
    for res in resultados_list:
        n = res['num_vertices']
        m = res['num_arestas']
        densidade = (2 * m) / (n * (n - 1)) if n > 1 else 0
        grau_medio = (2 * m) / n if n > 0 else 0
        
        info_instancias.append({
            'instancia': res['instancia_id'],
            'num_vertices': n,
            'num_arestas': m,
            'densidade': round(densidade, 4),
            'grau_medio': round(grau_medio, 2),
            'cores_encontradas': res['cores_heuristica'],
            'tempo_segundos': res['tempo_segundos']
        })
    
    df_info = pd.DataFrame(info_instancias)
    caminho = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'csv', 'informacoes_instancias.csv'
    )
    df_info.to_csv(caminho, index=False)
    print(f"✓ Informações das instâncias salvas em: informacoes_instancias.csv")

print("✓ Função salvar_informacoes_instancias definida")

✓ Função salvar_informacoes_instancias definida


## 11. Geração de Gráficos de Análise - Parte 1

### Objetivo desta Seção
Criar visualizações que demonstram o crescimento exponencial da força bruta e analisam o número cromático encontrado.

## Gráfico 1: Escalabilidade (Tempo vs Tamanho)

### Características
- **Eixo Y (log)**: Tempo em escala logarítmica (semilogy)
  - Revelam crescimento exponencial como reta
- **Eixo X (linear)**: Tamanho do grafo em vértices
- **Linha azul**: Tempo **médio** por tamanho
- **Área sombreada**: Intervalo min-max observado

### Interpretação
- **Inclinação suave**: Crescimento polinomial
- **Inclinação íngreme**: Crescimento exponencial
- **Espaçamento**: Variabilidade entre instâncias

### Propósito
Demonstra por que força bruta é inviável para n > ~15:
- n=10: ~0.001s
- n=13: ~1-10s
- n=15: ~1000s+ (proibitivo)

---

## Gráfico 2: Número Cromático vs Tamanho

### Características
- **Linha verde** com marcadores quadrados
- Mostra evolução de χ(G) com tamanho
- Linha de tendência implícita pela observação

### Observações Esperadas
- χ(G) não cresce linearmente com n
- Influência forte da densidade
- Máximo teórico: χ ≤ n

### Propósito
Validar que algoritmo encontra diferentes valores cromáticos
- Não é constante (grafo muda)
- Não cresce trivialmente

In [13]:
def gerar_graficos_parte1(resultados_list: List):
    """
    Gera gráficos de análise para Parte 1.
    
    Gera 2 gráficos separados:
    1. Gráfico de escalabilidade (tempo vs tamanho)
    2. Gráfico de número cromático (χ(G) vs tamanho)
    
    Args:
        resultados_list: Lista de resultados da força bruta
    """
    if not resultados_list:
        print("⚠ Sem dados para gerar gráficos Parte 1")
        return
    
    df = pd.DataFrame(resultados_list)
    
    # Agrupar por tamanho
    df_agrupado = df.groupby('tamanho').agg({
        'tempo_execucao': ['mean', 'min', 'max'],
        'num_cores': 'mean'
    }).reset_index()
    
    tamanhos = df_agrupado['tamanho'].values
    tempos_media = df_agrupado[('tempo_execucao', 'mean')].values
    tempos_min = df_agrupado[('tempo_execucao', 'min')].values
    tempos_max = df_agrupado[('tempo_execucao', 'max')].values
    cores_media = df_agrupado[('num_cores', 'mean')].values
    
    # Gráfico 1: Escalabilidade (Tempo vs Tamanho)
    plt.figure(figsize=(12, 7))
    
    plt.semilogy(tamanhos, tempos_media, 'o-', label='Média', linewidth=2.5, markersize=10, color='#2E86AB')
    plt.fill_between(tamanhos, tempos_min, tempos_max, alpha=0.25, label='Min-Max', color='#2E86AB')
    
    plt.xlabel('Tamanho do Grafo (número de vértices)', fontsize=13, fontweight='bold')
    plt.ylabel('Tempo de Execução (segundos)', fontsize=13, fontweight='bold')
    plt.title('Escalabilidade: Crescimento Exponencial do Tempo', fontsize=15, fontweight='bold')
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.legend(fontsize=11)
    
    plt.tight_layout()
    caminho1 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte1', 'graficos', 'grafico1_escalabilidade_tempo.png'
    )
    plt.savefig(caminho1, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Gráfico 1 salvo: grafico1_escalabilidade_tempo.png")
    
    # Gráfico 2: Número cromático vs Tamanho
    plt.figure(figsize=(12, 7))
    
    plt.plot(tamanhos, cores_media, 's-', color='#06A77D', linewidth=2.5, markersize=10, markeredgecolor='black', markeredgewidth=1.5)
    
    # Adicionar valores nos pontos
    for i, (x, y) in enumerate(zip(tamanhos, cores_media)):
        plt.text(x, y + 0.15, f'{y:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    plt.xlabel('Tamanho do Grafo (número de vértices)', fontsize=13, fontweight='bold')
    plt.ylabel('Número Cromático χ(G)', fontsize=13, fontweight='bold')
    plt.title('Número Cromático Encontrado', fontsize=15, fontweight='bold')
    plt.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    caminho2 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte1', 'graficos', 'grafico2_numero_cromatico.png'
    )
    plt.savefig(caminho2, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Gráfico 2 salvo: grafico2_numero_cromatico.png")
    
    print(f"\n✓ Todos os gráficos Parte 1 salvos em: resultados_2/parte1/graficos/")

print("✓ Função gerar_graficos_parte1 definida")

✓ Função gerar_graficos_parte1 definida


# Execução da Parte 2: Heurística Welsh-Powell nas Instâncias DIMACS

## Objetivo
Executar a heurística Welsh-Powell em 5 instâncias DIMACS:
- `le450_25a.col.txt`
- `inithx.i.1.col.txt`
- `r1000.1.col.txt`
- `ash958GPIA.col.txt`
- `wap03a.col.txt`

## Algoritmo Welsh-Powell
- **Estratégia**: Greedy com ordenação por grau decrescente
- **Complexidade**: O(n²)
- **Qualidade**: Bom custo-benefício para grafos gerais
- **Saída**: Número de cores e distribuição de cores por vértice

## Arquivos de Saída
- `resultados_2/parte2/csv/informacoes_instancias.csv` - Metadados das instâncias
- `resultados_2/parte2/csv/resultados_welsh_powell.csv` - Resultados detalhados
- Gráficos e visualizações em `resultados_2/parte2/grafos/` e `resultados_2/parte2/graficos/`


In [14]:
def gerar_graficos_parte2(resultados_list: List):
    """
    Gera gráficos de análise para Parte 2.
    
    Gera 4 gráficos separados:
    1. Número de cores por instância
    2. Tempo de execução por instância
    3. Número de vértices vs cores
    4. Densidade vs cores
    
    Args:
        resultados_list: Lista de resultados da heurística
    """
    if not resultados_list:
        print("⚠ Sem dados para gerar gráficos Parte 2")
        return
    
    df = pd.DataFrame(resultados_list)
    instancias = df['instancia_id'].values
    
    # Gráfico 1: Número de cores por instância
    plt.figure(figsize=(10, 6))
    cores = df['cores_heuristica'].values
    cores_sorted_idx = np.argsort(cores)
    
    plt.bar(instancias[cores_sorted_idx], cores[cores_sorted_idx], color='steelblue', edgecolor='black', linewidth=1.5)
    plt.xlabel('Instância DIMACS', fontsize=12, fontweight='bold')
    plt.ylabel('Número de Cores (Heurística)', fontsize=12, fontweight='bold')
    plt.title('Cores Encontradas por Instância', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    
    # Adicionar valores nas barras
    for idx in cores_sorted_idx:
        plt.text(instancias[idx], cores[idx] + 1, str(int(cores[idx])), 
                ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    plt.tight_layout()
    caminho1 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'graficos', 'grafico1_cores_por_instancia.png'
    )
    plt.savefig(caminho1, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Gráfico 1 salvo: grafico1_cores_por_instancia.png")
    
    # Gráfico 2: Tempo de execução por instância
    plt.figure(figsize=(10, 6))
    tempos = df['tempo_segundos'].values
    tempos_sorted_idx = np.argsort(tempos)
    
    plt.bar(instancias[tempos_sorted_idx], tempos[tempos_sorted_idx], color='coral', edgecolor='black', linewidth=1.5)
    plt.xlabel('Instância DIMACS', fontsize=12, fontweight='bold')
    plt.ylabel('Tempo (segundos)', fontsize=12, fontweight='bold')
    plt.title('Tempo de Execução por Instância', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    
    # Adicionar valores nas barras
    for idx in tempos_sorted_idx:
        plt.text(instancias[idx], tempos[idx] + max(tempos)*0.02, f'{tempos[idx]:.4f}s', 
                ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    plt.tight_layout()
    caminho2 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'graficos', 'grafico2_tempo_execucao.png'
    )
    plt.savefig(caminho2, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Gráfico 2 salvo: grafico2_tempo_execucao.png")
    
    # Gráfico 3: Número de vértices vs Cores
    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(df['num_vertices'], df['cores_heuristica'], s=300, alpha=0.7, 
               c=df['num_vertices'], cmap='viridis', edgecolors='black', linewidth=2)
    
    for i, txt in enumerate(df['instancia_id']):
        plt.annotate(txt, (df['num_vertices'].iloc[i], df['cores_heuristica'].iloc[i]),
                    fontsize=12, fontweight='bold', ha='center', va='center')
    
    plt.xlabel('Número de Vértices', fontsize=12, fontweight='bold')
    plt.ylabel('Número de Cores', fontsize=12, fontweight='bold')
    plt.title('Vértices vs Cores', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.colorbar(scatter, label='Número de Vértices')
    
    plt.tight_layout()
    caminho3 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'graficos', 'grafico3_vertices_vs_cores.png'
    )
    plt.savefig(caminho3, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Gráfico 3 salvo: grafico3_vertices_vs_cores.png")
    
    # Gráfico 4: Densidade vs Cores
    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(df['densidade'], df['cores_heuristica'], s=300, alpha=0.7,
               c=df['cores_heuristica'], cmap='Reds', edgecolors='black', linewidth=2)
    
    for i, txt in enumerate(df['instancia_id']):
        plt.annotate(txt, (df['densidade'].iloc[i], df['cores_heuristica'].iloc[i]),
                    fontsize=12, fontweight='bold', ha='center', va='center')
    
    plt.xlabel('Densidade do Grafo', fontsize=12, fontweight='bold')
    plt.ylabel('Número de Cores', fontsize=12, fontweight='bold')
    plt.title('Densidade vs Cores', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.colorbar(scatter, label='Número de Cores')
    
    plt.tight_layout()
    caminho4 = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'graficos', 'grafico4_densidade_vs_cores.png'
    )
    plt.savefig(caminho4, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Gráfico 4 salvo: grafico4_densidade_vs_cores.png")
    
    print(f"\n✓ Todos os gráficos Parte 2 salvos em: resultados_2/parte2/graficos/")

print("✓ Função gerar_graficos_parte2 definida")

✓ Função gerar_graficos_parte2 definida


In [15]:
def consolidar_e_salvar_dados_csv():
    """
    Consolida e valida todos os CSVs gerados no fluxo de execução.
    Verifica a origem e integridade dos dados antes de usar em gerar_tabelas_latex.
    
    Retorna:
        dict: Dicionário com informações sobre os CSVs encontrados e validados
    """
    
    print("\n" + "="*70)
    print("CONSOLIDANDO DADOS E SALVANDO EM CSV")
    print("="*70)
    
    # Criar diretório para resultados consolidados se não existir
    from pathlib import Path
    Path('resultados_2').mkdir(exist_ok=True)
    
    print("\n📊 ORIGEM DOS DADOS (CSVs gerados automaticamente):\n")
    
    print("PARTE 1 - FORÇA BRUTA:")
    print("  1️⃣  parametros_grafos.csv")
    print("      Origem: processar_instancias_por_tamanho()")
    print("      Conteúdo: Parâmetros estruturais dos 27 grafos aleatórios gerados")
    print("      (tamanho: 5-13 vértices, 3 instâncias cada)")
    print()
    print("  2️⃣  resultados_forca_bruta.csv")
    print("      Origem: algoritmo_forca_bruta_coloracao()")
    print("      Conteúdo: Número cromático encontrado e tempo para cada grafo")
    print()
    
    print("PARTE 2 - HEURÍSTICA WELSH-POWELL:")
    print("  3️⃣  resultados_heuristica.csv")
    print("      Origem: processar_instancias_dimacs() → algoritmo_welsh_powell()")
    print("      Conteúdo: Cores encontradas, tempo e parametros das 5 instâncias DIMACS")
    print()
    print("  4️⃣  informacoes_instancias.csv")
    print("      Origem: salvar_informacoes_instancias()")
    print("      Conteúdo: Informações consolidadas das instâncias DIMACS")
    print("      (densidade, grau médio, cores, tempo)")
    print()
    
    # Verificar quais arquivos CSV foram gerados
    import os
    csv_files = []
    csv_info = {}
    
    print("-" * 70)
    print("Verificando arquivos CSV gerados...\n")
    
    for root, dirs, files in os.walk('resultados_2'):
        for file in sorted(files):
            if file.endswith('.csv'):
                path = os.path.join(root, file)
                csv_files.append(path)
                try:
                    df = pd.read_csv(path)
                    csv_info[file] = {
                        'path': path,
                        'linhas': len(df),
                        'colunas': len(df.columns),
                        'dataframe': df,
                        'status': '✓'
                    }
                    print(f"✓ {path}")
                    print(f"    Linhas: {len(df)} | Colunas: {len(df.columns)}")
                    print(f"    Primeiras linhas:")
                    print(df.head(2).to_string(index=False).replace('\n', '\n    '))
                    print()
                except Exception as e:
                    csv_info[file] = {
                        'path': path,
                        'status': '⚠️',
                        'erro': str(e)
                    }
                    print(f"⚠️  {path} - Erro: {e}\n")
    
    if not csv_files:
        print("  ⚠️  Nenhum CSV encontrado. Execute as células de solução primeiro!")
        return {
            'status': 'erro',
            'mensagem': 'Nenhum CSV gerado',
            'arquivos': []
        }
    else:
        print(f"✅ Total de arquivos CSV gerados: {len(csv_files)}")
    
    print("\n" + "="*70)
    print("PRÓXIMAS ETAPAS")
    print("="*70)
    
    print("""
1. Abra o notebook: gerar_tabelas_latex.ipynb
2. Execute as células para gerar as tabelas LaTeX
3. Os CSVs acima serão automaticamente lidos e convertidos em tabelas

📁 Arquivos CSV para usar em gerar_tabelas_latex.ipynb:
""")
    
    for csv_path in sorted(csv_files):
        print(f"   - {csv_path}")
    
    print("""
Este fluxo garante que:
✓ Dados são calculados uma única vez (em solucao_coloracao_completa)
✓ CSVs armazenam os resultados (fonte de verdade única)
✓ Tabelas LaTeX são geradas automaticamente (sempre sincronizadas com dados)
""")
    
    return {
        'status': 'sucesso',
        'total_arquivos': len(csv_files),
        'arquivos': csv_files,
        'informacoes': csv_info
    }


# Resumo da Solução Completa - Notebook 2 (2 Algoritmos)

## 🎯 Objetivo
Este notebook implementa a **Solução Completa** para o Problema da Coloração de Grafos com **2 ALGORITMOS**:
- **Parte 1**: Força Bruta em grafos aleatórios (5-13 vértices)
- **Parte 2**: Heurística Welsh-Powell em instâncias DIMACS

📌 **NOTA**: Para análise com **3 algoritmos** (incluindo DSATUR), use o notebook `solucao_coloracao_completa_1.ipynb`.

## 📊 Análises Realizadas

### ANÁLISE 1: FORÇA BRUTA
- **Instâncias**: Grafos aleatórios de **5 a 15 vértices** (3 instâncias por tamanho = **33 grafos**)
- **Algoritmo**: Força bruta determinística (encontra χ(G) exato)
- **Complexidade**: Exponencial O(k^n) - grafos maiores levam MUITO tempo
- **Saída**: `resultados_2/parte1/csv/resultados_forca_bruta.csv`

⚠️ **NOTA IMPORTANTE**: 
- O trabalho especifica "tamanho máximo vai girar em torno de **13 a 20 cidades**"
- Configuração atual: **5-15 vértices** (pode ajustar até 20 se o tempo permitir)
- Grafos com 16+ vértices podem levar **horas** para processar!
### ANÁLISE 2: HEURÍSTICA WELSH-POWELL (APENAS)

### ANÁLISE 2: HEURÍSTICA WELSH-POWELL
- **Instâncias**: 5 grafos DIMACS (le450, inithx, r1000, ash958, wap03a)
- **Algoritmo**: Welsh-Powell (greedy por grau decrescente)
- **Comparação**: 2 algoritmos (Força Bruta vs Welsh-Powell)

📌 **Para comparação com DSATUR**, veja `solucao_coloracao_completa_1.ipynb` que implementa 3 algoritmos.

## 📁 Estrutura de Saída

```
resultados_2/
├── parte1/           (Força Bruta - 33 instâncias)
│   ├── csv/
│   │   ├── parametros_grafos.csv       (Parâmetros estruturais)
│   │   └── resultados_forca_bruta.csv  (χ(G) e tempo)
│   ├── grafos/       (Visualizações dos grafos coloridos)
│   └── graficos/     (Gráficos de escalabilidade)
└── parte2/           (Welsh-Powell - 5 instâncias DIMACS)
    ├── csv/
    │   ├── informacoes_instancias.csv
    │   ├── resultados_welsh_powell.csv
    │   └── comparacao_algoritmos.csv   (Força Bruta vs Welsh-Powell)
    ├── grafos/       (Visualizações dos grafos coloridos)
    └── graficos/     (Gráficos comparativos)
```

## 📈 Fluxo de Execução
1. **Célula 1-8**: Setup e configuração
2. **Célula 9-31**: Definição dos algoritmos
3. **Célula 32**: Funções wrapper para geração de instâncias
4. **Célula 33-34**: Funções de consolidação e execução principal
5. **Célula 38**: **EXECUTAR SOLUÇÃO COMPLETA** ⚡

## ⏱️ Tempo de Execução Estimado
- **5-10 vértices**: Segundos (< 1 min)
- **11-13 vértices**: Minutos (1-5 min)
- **14-15 vértices**: Minutos (5-20 min)
- **16-20 vértices**: Horas (pode ser inviável)
## ✅ Validação
## ✅ Validação
Todos os CSVs são gerados automaticamente e podem ser usados por `gerar_tabelas_latex.ipynb` para criar as tabelas do relatório final.

## 🔧 Como Ajustar o Intervalo
Para testar grafos maiores, modifique a chamada na **célula 38**:
```python
# Padrão (5-15 vértices)
executar_solucao_completa()

# Personalizado (exemplo: 5-18 vértices)

# executar_solucao_completa(tamanho_max=18)
```

```

In [16]:
def gerar_instancias_aleatorias(tamanho_min: int = 5, tamanho_max: int = 15, 
                                instancias_por_tamanho: int = 3) -> List:
    """
    Wrapper que gera instâncias aleatórias de grafos.
    
    REQUISITO: O trabalho especifica "tamanho máximo vai girar em torno de 13 a 20"
    Intervalo padrão: 5-15 vértices (33 instâncias, 3 por tamanho)
    
    Args:
        tamanho_min: Tamanho mínimo (padrão: 5 vértices)
        tamanho_max: Tamanho máximo (padrão: 15 vértices - pode tentar até 20)
        instancias_por_tamanho: Quantas instâncias por tamanho (padrão: 3)
    
    Retorna:
        Lista de dicionários com parâmetros e grafos gerados
    """
    instancias = []
    
    print(f"\n📊 Gerando instâncias aleatórias: {tamanho_min}-{tamanho_max} vértices")
    print(f"   Total esperado: {(tamanho_max - tamanho_min + 1) * instancias_por_tamanho} instâncias\n")
    
    for tamanho in range(tamanho_min, tamanho_max + 1):
        for i in range(instancias_por_tamanho):
            seed = tamanho * 100 + i
            # Probabilidade aumenta com tamanho (grafos maiores = mais densos)
            prob = 0.3 + (tamanho - tamanho_min) * 0.05
            
            grafo = gerar_instancia_grafo_aleatorio(
                num_vertices=tamanho,
                probabilidade=min(prob, 0.9),
                seed=seed
            )
            
            instancias.append({
                'tamanho': tamanho,
                'grafo': grafo,
                'probabilidade': prob,
                'seed': seed
            })
    
    return instancias


def processar_instancias_aleatorias(instancias: List) -> Tuple[List, List]:
    """
    Processa instâncias aleatórias usando força bruta.
    
    IMPORTANTE: Força bruta tem complexidade exponencial!
    - Grafos até 13 vértices: Segundos
    - Grafos 14-15 vértices: Minutos
    - Grafos 16-20 vértices: Horas (pode ser inviável)
    
    Args:
        instancias: Lista de dicionários com grafos
        
    Returns:
        Tupla (parametros_list, resultados_list)
    """
    parametros_list = []
    resultados_list = []
    
    total = len(instancias)
    print(f"\n🔄 Processando {total} instâncias com Força Bruta...")
    print("⚠️  ATENÇÃO: Grafos maiores podem levar MUITO tempo!\n")
    
    for i, instancia in enumerate(instancias, 1):
        grafo = instancia['grafo']
        tamanho = instancia['tamanho']
        
        # Calcular parâmetros estruturais
        parametros = {
            'id_instancia': i,
            'tamanho': tamanho,
            'num_vertices': grafo.number_of_nodes(),
            'num_arestas': grafo.number_of_edges(),
            'densidade': nx.density(grafo),
            'grau_medio': sum(dict(grafo.degree()).values()) / grafo.number_of_nodes() if grafo.number_of_nodes() > 0 else 0
        }
        parametros_list.append(parametros)
        
        # Executar força bruta
        try:
            print(f"  [{i}/{total}] Tamanho {tamanho}: ", end="", flush=True)
            
            numero_cromatico, coloracao, tempo = algoritmo_forca_bruta_coloracao(grafo)
            
            resultado = {
                'id_instancia': i,
                'tamanho': tamanho,
                'num_vertices': grafo.number_of_nodes(),
                'num_arestas': grafo.number_of_edges(),
                'num_cores': numero_cromatico,
                'tempo_execucao': tempo,
                'densidade': nx.density(grafo),
                'coloracao': coloracao,
                'grafo': grafo
            }
            resultados_list.append(resultado)
            
            print(f"χ(G)={numero_cromatico}, tempo={tempo:.4f}s ✓")
                
        except Exception as e:
            print(f"⚠️ ERRO: {str(e)}")
    
    return parametros_list, resultados_list

print("✓ Funções gerar_instancias_aleatorias e processar_instancias_aleatorias definidas")
print("  Intervalo padrão: 5-15 vértices (pode ajustar até 20 se tempo permitir)")


✓ Funções gerar_instancias_aleatorias e processar_instancias_aleatorias definidas
  Intervalo padrão: 5-15 vértices (pode ajustar até 20 se tempo permitir)


In [17]:
def consolidar_e_salvar_dados_csv():  # type: ignore
    """
    Consolida e valida todos os CSVs gerados no fluxo de execução.
    Verifica a origem e integridade dos dados antes de usar em gerar_tabelas_latex.
    
    Inclui as 2 análises principais:
    1. Força Bruta (instâncias aleatórias)
    2. Heurística Welsh-Powell (instâncias DIMACS)
    
    Retorna:
        dict: Dicionário com informações sobre os CSVs encontrados e validados
    """
    
    print("\n" + "="*70)
    print("CONSOLIDANDO DADOS E SALVANDO EM CSV")
    print("="*70)
    
    print("\n📊 ORIGEM DOS DADOS (CSVs gerados automaticamente):\n")
    
    print("PARTE 1 - FORÇA BRUTA (Instâncias Aleatórias):")
    print("  1️⃣  parametros_grafos.csv")
    print("      Origem: processar_instancias_aleatorias()")
    print("      Conteúdo: Parâmetros estruturais dos 27 grafos aleatórios gerados")
    print("      (tamanho: 5-13 vértices, 3 instâncias cada)")
    print()
    print("  2️⃣  resultados_forca_bruta.csv")
    print("      Origem: algoritmo_forca_bruta_coloracao()")
    print("      Conteúdo: Número cromático encontrado e tempo para cada grafo")
    print()
    
    print("PARTE 2 - HEURÍSTICA WELSH-POWELL (Instâncias DIMACS):")
    print("  3️⃣  resultados_welsh_powell.csv")
    print("      Origem: processar_instancias_dimacs() → algoritmo_welsh_powell()")
    print("      Conteúdo: Cores encontradas, tempo e parâmetros das 5 instâncias DIMACS")
    print()
    
    print("COMPARAÇÃO:")
    print("  4️⃣  comparacao_algoritmos.csv")
    print("      Origem: executar_solucao_completa()")
    print("      Conteúdo: Resumo comparativo entre Força Bruta e Welsh-Powell")
    print()
    
    print("INFORMAÇÕES CONSOLIDADAS:")
    print("  5️⃣  informacoes_instancias.csv")
    print("      Origem: salvar_informacoes_instancias()")
    print("      Conteúdo: Informações consolidadas das instâncias DIMACS")
    print("      (densidade, grau médio, cores, tempo)")
    print()
    
    # Verificar quais arquivos CSV foram gerados
    import os
    csv_files = []
    csv_info = {}
    
    print("-" * 70)
    print("Verificando arquivos CSV gerados em resultados_2...\n")
    
    for root, dirs, files in os.walk('resultados_2'):
        for file in sorted(files):
            if file.endswith('.csv'):
                path = os.path.join(root, file)
                csv_files.append(path)
                try:
                    df = pd.read_csv(path)
                    csv_info[file] = {
                        'path': path,
                        'linhas': len(df),
                        'colunas': len(df.columns),
                        'dataframe': df,
                        'status': '✓'
                    }
                    print(f"✓ {path}")
                    print(f"    Linhas: {len(df)} | Colunas: {len(df.columns)}")
                    print(f"    Primeiras linhas:")
                    print(df.head(2).to_string(index=False).replace('\n', '\n    '))
                    print()
                except Exception as e:
                    csv_info[file] = {
                        'path': path,
                        'status': '⚠️',
                        'erro': str(e)
                    }
                    print(f"⚠️  {path} - Erro: {e}\n")
    
    if not csv_files:
        print("  ⚠️  Nenhum CSV encontrado em resultados_2/. Execute as células de solução primeiro!")
        return {
            'status': 'erro',
            'mensagem': 'Nenhum CSV gerado',
            'arquivos': []
        }
    else:
        print(f"✅ Total de arquivos CSV gerados: {len(csv_files)}")
    
    print("\n" + "="*70)
    print("PRÓXIMAS ETAPAS")
    print("="*70)
    
    print("""
1. Abra o notebook: gerar_tabelas_latex.ipynb
2. Execute as células para gerar as tabelas LaTeX
3. Os CSVs acima serão automaticamente lidos e convertidos em tabelas

📁 Arquivos CSV para usar em gerar_tabelas_latex.ipynb:
""")
    
    for csv_path in sorted(csv_files):
        print(f"   - {csv_path}")
    
    print("""
Este fluxo garante que:
✓ Dados são calculados uma única vez (em solucao_coloracao_completa)
✓ CSVs armazenam os resultados (fonte de verdade única)
✓ Tabelas LaTeX são geradas automaticamente (sempre sincronizadas com dados)
✓ Comparação dos 2 algoritmos disponível para análise
""")
    
    return {
        'status': 'sucesso',
        'total_arquivos': len(csv_files),
        'arquivos': csv_files,
        'informacoes': csv_info
    }

print("✓ Função consolidar_e_salvar_dados_csv definida")

✓ Função consolidar_e_salvar_dados_csv definida


In [ ]:
def executar_solucao_completa(tamanho_min: int = 5, tamanho_max: int = 15, verbose: bool = False):  # type: ignore
    """
    Orquestra a execução completa da solução seguindo a ordem:
    
    PARTE 1 (Força Bruta):
    1. Processar instâncias
    2. Salvar CSVs
    3. Gerar gráficos de análise
    4. Salvar visualizações dos grafos
    
    PARTE 2 (Welsh-Powell):
    1. Processar instâncias DIMACS
    2. Salvar CSVs
    3. Gerar gráficos de análise
    4. Salvar visualizações dos grafos
    
    FINAL:
    - CSV de comparação entre os 2 algoritmos
    
    Args:
        tamanho_min: Tamanho mínimo dos grafos (padrão: 5)
        tamanho_max: Tamanho máximo dos grafos (padrão: 15, pode ir até 20)
        verbose: Se True, mostra prints detalhados (padrão: False)
    """
    
    # 📍 REGISTRAR HORA DE INÍCIO
    from datetime import datetime
    hora_inicio = datetime.now()
    
    print("\n" + "="*80)
    print("🕐 INICIANDO EXECUÇÃO DA SOLUÇÃO COMPLETA")
    print("="*80)
    print(f"⏰ Data e Hora: {hora_inicio.strftime('%d/%m/%Y às %H:%M:%S')}")
    print(f"📊 Configuração: tamanho_min={tamanho_min}, tamanho_max={tamanho_max}")
    print(f"📈 Total de instâncias esperadas: {(tamanho_max - tamanho_min + 1) * 3 + 5} grafos")
    print("="*80)
    
    if verbose:
        print("\n" + "="*80)
        print("SOLUÇÃO COMPLETA - COLORAÇÃO DE GRAFOS (FORÇA BRUTA + WELSH-POWELL)")
        print("="*80)
        print(f"Configuração: Grafos aleatórios de {tamanho_min} a {tamanho_max} vértices")
        print(f"Total de instâncias Parte 1: {(tamanho_max - tamanho_min + 1) * 3} grafos")
        print("="*80)
    
    # ========================================================================
    # PARTE 1: FORÇA BRUTA
    # ========================================================================
    if verbose:
        print("\n" + "="*80)
        print("PARTE 1: FORÇA BRUTA EM INSTÂNCIAS ALEATÓRIAS")
        print("="*80)
        print("\n[1.1] Processando instâncias aleatórias...")
        print("-" * 80)
    
    print("\n[1.1] Processando instâncias aleatórias...")
    print("-" * 80)
    
    # Gerar e processar instâncias aleatórias
    instancias = gerar_instancias_aleatorias(tamanho_min, tamanho_max)
    parametros_list, resultados_list = processar_instancias_aleatorias(instancias)
    
    if verbose:
        print(f"✓ {len(resultados_list)} instâncias processadas")
        print("\n[1.2] Salvando CSVs Parte 1...")
        print("-" * 80)
    print("\n[1.2] Salvando CSVs Parte 1...")
    print("-" * 80)
    df_params = pd.DataFrame(parametros_list)
    caminho_params = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte1', 'csv', 'parametros_grafos.csv'
    )
    df_params.to_csv(caminho_params, index=False)
    
    df_fb = pd.DataFrame([{k: v for k, v in r.items() if k not in ['coloracao', 'grafo']} 
                          for r in resultados_list])
    caminho_fb = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte1', 'csv', 'resultados_forca_bruta.csv'
    )
    df_fb.to_csv(caminho_fb, index=False)
    
    if verbose:
        print(f"✓ Parâmetros salvos: {caminho_params}")
        print(f"✓ Resultados força bruta salvos: {caminho_fb}")
        print("\n[1.3] Gerando gráficos de análise Parte 1...")
        print("-" * 80)
    print("\n[1.3] Gerando gráficos de análise Parte 1...")
    print("-" * 80)
    gerar_graficos_parte1([{k: v for k, v in r.items() if k not in ['coloracao', 'grafo']}
                           for r in resultados_list])  # type: ignore
    if verbose:
        print("\n[1.4] Salvando visualizações dos grafos Parte 1...")
        print("-" * 80)
    print("\n[1.4] Salvando visualizações dos grafos Parte 1...")
    print("-" * 80)
    for resultado in resultados_list:
        if 'grafo' in resultado and 'coloracao' in resultado:
            id_inst = resultado['id_instancia']
            tamanho = resultado['tamanho']
            caminho_grafo = ConfiguradorDiretorios.obter_caminho_arquivo(
                'parte1', 'grafos', f"grafo_inst{id_inst:02d}_n{tamanho:02d}.png"
            )
            visualizar_e_salvar_grafo(resultado["grafo"], resultado["coloracao"], caminho_grafo)

    
    if verbose:
        print(f"✓ {len(resultados_list)} visualizações de grafos salvas em resultados_2/parte1/grafos/")
    
    # ========================================================================
    # PARTE 2: WELSH-POWELL
    # ========================================================================
    if verbose:
        print("\n" + "="*80)
        print("PARTE 2: HEURÍSTICA WELSH-POWELL EM INSTÂNCIAS DIMACS")
        print("="*80)
        print("\n[2.1] Processando instâncias DIMACS...")
        print("-" * 80)
    
    print("\n[2.1] Processando instâncias DIMACS...")
    print("-" * 80)
    
    # Processar instâncias DIMACS
    resultados_dimacs = processar_instancias_dimacs()
    
    if verbose:
        print(f"✓ {len(resultados_dimacs)} instâncias DIMACS processadas")
        print("\n[2.2] Salvando CSVs Parte 2...")
        print("-" * 80)
    print("-" * 80)
    df_wp = pd.DataFrame([{k: v for k, v in r.items() if k not in ['coloracao', 'grafo']} 
                          for r in resultados_dimacs])
    caminho_wp = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'csv', 'resultados_welsh_powell.csv'
    )
    df_wp.to_csv(caminho_wp, index=False)
    print(f"✓ Resultados Welsh-Powell salvos: {caminho_wp}")
    
    df_info = pd.DataFrame([{
        'instancia': r['instancia_id'],
        'num_vertices': r['num_vertices'],
        'num_arestas': r['num_arestas'],
        'densidade': round(r['densidade'], 4),
        'grau_medio': round(r['grau_medio'], 2),
        'cores_encontradas': r['cores_heuristica'],
        'tempo_segundos': r['tempo_segundos']
    } for r in resultados_dimacs])
    caminho_info = ConfiguradorDiretorios.obter_caminho_arquivo(
        'parte2', 'csv', 'informacoes_instancias.csv'
    )
    df_info.to_csv(caminho_info, index=False)
    print(f"✓ Informações instâncias salvos: {caminho_info}")
    
    # 2.3 Gerar gráficos de análise
    print("\n[2.3] Gerando gráficos de análise Parte 2...")
    print("-" * 80)
    gerar_graficos_parte2([{k: v for k, v in r.items() if k not in ['coloracao', 'grafo']}
                           for r in resultados_dimacs])  # type: ignore
    if verbose:
        print("\n[2.4] Salvando visualizações dos grafos DIMACS...")
        print("-" * 80)
    print("\n[2.4] Salvando visualizações dos grafos DIMACS...")
    print("-" * 80)
    for resultado in resultados_dimacs:
        if 'grafo' in resultado and 'coloracao' in resultado:
            inst_id = resultado['instancia_id']
            caminho_grafo = ConfiguradorDiretorios.obter_caminho_arquivo(
                'parte2', 'grafos', f"instancia_{inst_id}.png"
            )
            visualizar_e_salvar_grafo(resultado["grafo"], resultado["coloracao"], caminho_grafo)
    
    if verbose:
        print(f"✓ {len(resultados_dimacs)} visualizações de grafos DIMACS salvas em resultados_2/parte2/grafos/")
    print(f"✓ {len(resultados_dimacs)} visualizações de grafos DIMACS salvas em resultados_2/parte2/grafos/")
    
    # FINAL: COMPARAÇÃO
    # FINAL: COMPARAÇÃO
    # ========================================================================
    if verbose:
        print("\n" + "="*80)
        print("COMPARAÇÃO ENTRE OS 2 ALGORITMOS")
        print("="*80)
        print("\n[3] Gerando CSV de comparação...")
        print("-" * 80)
    df_comparacao = pd.DataFrame({
        'Algoritmo': ['Força Bruta', 'Welsh-Powell'],
        'Instâncias': [len(df_fb), len(df_wp)],
        'Cores Médias': [
            f"{df_fb['num_cores'].mean():.2f}" if len(df_fb) > 0 else "N/A",
            f"{df_wp['cores_heuristica'].mean():.2f}" if len(df_wp) > 0 else "N/A"
        ],
        'Cores Mín.': [
            int(df_fb['num_cores'].min()) if len(df_fb) > 0 else "N/A",
            int(df_wp['cores_heuristica'].min()) if len(df_wp) > 0 else "N/A"
        ],
        'Cores Máx.': [
            int(df_fb['num_cores'].max()) if len(df_fb) > 0 else "N/A",
            int(df_wp['cores_heuristica'].max()) if len(df_wp) > 0 else "N/A"
        ],
        'Tempo Médio (s)': [
            f"{df_fb['tempo_execucao'].mean():.4f}" if len(df_fb) > 0 else "N/A",
            f"{df_wp['tempo_segundos'].mean():.4f}" if len(df_wp) > 0 else "N/A"
        ],
        'Qualidade': ['Ótima (χ exato)', 'Aproximada'],
        'Complexidade': ['Exponencial', 'O(n²)']
    })
    
    caminho_comp = 'resultados_2/comparacao_algoritmos.csv'
    df_comparacao.to_csv(caminho_comp, index=False)
    
    if verbose:
        print(f"✓ CSV de comparação salvo: {caminho_comp}")
        print("\nResumo da Comparação:")
        print(df_comparacao.to_string(index=False))
        
        # ========================================================================
        # RESUMO FINAL
        # ========================================================================
        print("\n" + "="*80)
        print("RESUMO DA EXECUÇÃO COMPLETA")
        print("="*80)
        
        print(f"\n✅ PARTE 1 - FORÇA BRUTA:")
        print(f"  • Instâncias: {len(df_fb)} grafos ({tamanho_min}-{tamanho_max} vértices)")
        print(f"  • CSV resultados: {caminho_fb}")
        print(f"  • CSV parâmetros: {caminho_params}")
        print(f"  • Gráficos: resultados_2/parte1/graficos/ (2 arquivos)")
        print(f"  • Visualizações: resultados_2/parte1/grafos/ ({len(resultados_list)} arquivos)")
        print(f"  • Cores: {int(df_fb['num_cores'].min())} a {int(df_fb['num_cores'].max())}")
        print(f"  • Tempo total: {df_fb['tempo_execucao'].sum():.2f}s")
        
        print(f"\n✅ PARTE 2 - WELSH-POWELL:")
        print(f"  • Instâncias: {len(df_wp)} grafos DIMACS")
        print(f"  • CSV resultados: {caminho_wp}")
        print(f"  • CSV informações: {caminho_info}")
        print(f"  • Gráficos: resultados_2/parte2/graficos/ (4 arquivos)")
        print(f"  • Visualizações: resultados_2/parte2/grafos/ ({len(resultados_dimacs)} arquivos)")
        print(f"  • Cores: {int(df_wp['cores_heuristica'].min())} a {int(df_wp['cores_heuristica'].max())}")
        
        print(f"\n✅ COMPARAÇÃO FINAL:")
        print(f"  • CSV comparação: {caminho_comp}")
        print(f"  • Status: Análise completa dos 2 algoritmos")
        
        print("\n" + "="*80)
        print("🎉 EXECUÇÃO COMPLETA FINALIZADA COM SUCESSO!")
        print("="*80)
    
    # 📍 REGISTRAR HORA DE FIM E CALCULAR TEMPO TOTAL
    hora_fim = datetime.now()
    tempo_total = hora_fim - hora_inicio
    
    print("\n" + "="*80)
    print("✅ EXECUÇÃO FINALIZADA")
    print("="*80)
    print(f"⏰ Horário de início:  {hora_inicio.strftime('%d/%m/%Y às %H:%M:%S')}")
    print(f"⏰ Horário de término: {hora_fim.strftime('%d/%m/%Y às %H:%M:%S')}")
    print(f"⏱️  Tempo total:        {str(tempo_total).split('.')[0]}")
    print(f"   ({tempo_total.total_seconds():.0f} segundos = {tempo_total.total_seconds()/60:.1f} minutos)")
    print("="*80)
    
    return {
        'forca_bruta': df_fb,
        'welsh_powell': df_wp,
        'comparacao': df_comparacao,
        'hora_inicio': hora_inicio,
        'hora_fim': hora_fim,
        'tempo_total': tempo_total
    }
print("  Use: executar_solucao_completa() para execução silenciosa")
print("  Use: executar_solucao_completa(verbose=True) para prints detalhados")
print("  Use: executar_solucao_completa(tamanho_max=18) para grafos maiores")



print("  Use: executar_solucao_completa(tamanho_max=18) para grafos maiores")

print("  Use: executar_solucao_completa(tamanho_max=18) para grafos maiores")
print("  Use: executar_solucao_completa(tamanho_max=18) para grafos maiores")


  Use: executar_solucao_completa() para execução silenciosa
  Use: executar_solucao_completa(verbose=True) para prints detalhados
  Use: executar_solucao_completa(tamanho_max=18) para grafos maiores
  Use: executar_solucao_completa(tamanho_max=18) para grafos maiores
  Use: executar_solucao_completa(tamanho_max=18) para grafos maiores
  Use: executar_solucao_completa(tamanho_max=18) para grafos maiores


def consolidar_e_salvar_dados_csv():
    """
    Consolida e valida todos os CSVs gerados no fluxo de execução.
    Verifica a origem e integridade dos dados antes de usar em gerar_tabelas_latex.
    
    Inclui as 2 análises principais:
    1. Força Bruta (instâncias aleatórias)
    2. Heurística Welsh-Powell (instâncias DIMACS)
    
    Retorna:
        dict: Dicionário com informações sobre os CSVs encontrados e validados
    """
    
    print("\n" + "="*70)
    print("CONSOLIDANDO DADOS E SALVANDO EM CSV")
    print("="*70)
    
    print("\n📊 ORIGEM DOS DADOS (CSVs gerados automaticamente):\n")
    
    print("PARTE 1 - FORÇA BRUTA (Instâncias Aleatórias):")
    print("  1️⃣  parametros_grafos.csv")
    print("      Origem: processar_instancias_aleatorias()")
    print("      Conteúdo: Parâmetros estruturais dos 27 grafos aleatórios gerados")
    print("      (tamanho: 5-13 vértices, 3 instâncias cada)")
    print()
    print("  2️⃣  resultados_forca_bruta.csv")
    print("      Origem: algoritmo_forca_bruta_coloracao()")
    print("      Conteúdo: Número cromático encontrado e tempo para cada grafo")
    print()
    
    print("PARTE 2 - HEURÍSTICA WELSH-POWELL (Instâncias DIMACS):")
    print("  3️⃣  resultados_welsh_powell.csv")
    print("      Origem: processar_instancias_dimacs() → algoritmo_welsh_powell()")
    print("      Conteúdo: Cores encontradas, tempo e parâmetros das 5 instâncias DIMACS")
    print()
    
    print("COMPARAÇÃO:")
    print("  4️⃣  comparacao_algoritmos.csv")
    print("      Origem: executar_solucao_completa()")
    print("      Conteúdo: Resumo comparativo entre Força Bruta e Welsh-Powell")
    print()
    
    print("INFORMAÇÕES CONSOLIDADAS:")
    print("  5️⃣  informacoes_instancias.csv")
    print("      Origem: salvar_informacoes_instancias()")
    print("      Conteúdo: Informações consolidadas das instâncias DIMACS")
    print("      (densidade, grau médio, cores, tempo)")
    print()
    
    # Verificar quais arquivos CSV foram gerados
    import os
    csv_files = []
    csv_info = {}
    
    print("-" * 70)
    print("Verificando arquivos CSV gerados em resultados_2...\n")
    
    for root, dirs, files in os.walk('resultados_2'):
        for file in sorted(files):
            if file.endswith('.csv'):
                path = os.path.join(root, file)
                csv_files.append(path)
                try:
                    df = pd.read_csv(path)
                    csv_info[file] = {
                        'path': path,
                        'linhas': len(df),
                        'colunas': len(df.columns),
                        'dataframe': df,
                        'status': '✓'
                    }
                    print(f"✓ {path}")
                    print(f"    Linhas: {len(df)} | Colunas: {len(df.columns)}")
                    print(f"    Primeiras linhas:")
                    print(df.head(2).to_string(index=False).replace('\n', '\n    '))
                    print()
                except Exception as e:
                    csv_info[file] = {
                        'path': path,
                        'status': '⚠️',
                        'erro': str(e)
                    }
                    print(f"⚠️  {path} - Erro: {e}\n")
    
    if not csv_files:
        print("  ⚠️  Nenhum CSV encontrado em resultados_2/. Execute as células de solução primeiro!")
        return {
            'status': 'erro',
            'mensagem': 'Nenhum CSV gerado',
            'arquivos': []
        }
    else:
        print(f"✅ Total de arquivos CSV gerados: {len(csv_files)}")
    
    print("\n" + "="*70)
    print("PRÓXIMAS ETAPAS")
    print("="*70)
    
    print("""
1. Abra o notebook: gerar_tabelas_latex.ipynb
2. Execute as células para gerar as tabelas LaTeX
3. Os CSVs acima serão automaticamente lidos e convertidos em tabelas

📁 Arquivos CSV para usar em gerar_tabelas_latex.ipynb:
""")
    
    for csv_path in sorted(csv_files):
        print(f"   - {csv_path}")
    
    print("""
Este fluxo garante que:
✓ Dados são calculados uma única vez (em solucao_coloracao_completa)
✓ CSVs armazenam os resultados (fonte de verdade única)
✓ Tabelas LaTeX são geradas automaticamente (sempre sincronizadas com dados)
""")
    
    return {
        'status': 'sucesso',
        'total_arquivos': len(csv_files),
        'arquivos': csv_files,
        'informacoes': csv_info
    }

print("✓ Função consolidar_e_salvar_dados_csv definida")

## 16. Consolidação e Validação de Resultados

### Objetivo desta Seção

Após a execução completa, consolidar todos os CSVs gerados e validar sua integridade antes de usar em relatórios LaTeX.

### Funcionalidades da Consolidação

A função `consolidar_e_salvar_dados_csv()` realiza:

1. **Varredura automática** de todos os arquivos CSV em `resultados_2/`
2. **Validação de integridade** - verifica se os arquivos podem ser lidos
3. **Exibição de estatísticas** - mostra linhas, colunas e primeiras linhas de cada CSV
4. **Confirmação de origem** - documenta de onde vieram os dados
5. **Preparação para LaTeX** - lista arquivos prontos para `gerar_tabelas_latex.ipynb`

### Análises Consolidadas

- **Força Bruta**: Demonstra crescimento exponencial do tempo com tamanho
- **Heurística Welsh-Powell**: Algoritmo guloso O(n+m) rápido
- **Heurística DSATUR**: Algoritmo adaptativo com melhor qualidade
- **Informações DIMACS**: Metadados consolidados das 5 instâncias

## Resumo Executivo - Notebook 2

Este notebook implementa:
- ✅ **Força Bruta**: Análise completa com 27 instâncias aleatórias
- ✅ **Welsh-Powell**: Heurística greedy nas 5 instâncias DIMACS

Todos os resultados são salvos automaticamente em `resultados_2/` com estrutura organizada (parte1/parte2 com csv/grafos/graficos).


In [19]:
# Executar a solução completa com tamanho máximo 20
# tamanho_max=20 gera 48 instâncias (16 tamanhos × 3 instâncias cada)
# 
# ESTIMATIVA DE TEMPO:
# - Grafos pequenos (5-12): < 1 segundo (instantâneo)
# - Grafos médios (13-16): 1-30 segundos
# - Grafos maiores (17-20): 30 segundos a 5-10 minutos por grafo (depende da densidade)
#
# Tempo total estimado: 10-30 minutos (pode variar bastante)
# 
# Nota: O tempo aumenta exponencialmente. Se demorar muito, considere reduzir para 15-18.
executar_solucao_completa(tamanho_max=20)


[1.1] Processando instâncias aleatórias...
--------------------------------------------------------------------------------

📊 Gerando instâncias aleatórias: 5-20 vértices
   Total esperado: 48 instâncias


🔄 Processando 48 instâncias com Força Bruta...
⚠️  ATENÇÃO: Grafos maiores podem levar MUITO tempo!

  [1/48] Tamanho 5: χ(G)=2, tempo=0.0000s ✓
  [2/48] Tamanho 5: χ(G)=2, tempo=0.0000s ✓
  [3/48] Tamanho 5: χ(G)=2, tempo=0.0000s ✓
  [4/48] Tamanho 6: χ(G)=2, tempo=0.0000s ✓
  [5/48] Tamanho 6: χ(G)=2, tempo=0.0000s ✓
  [6/48] Tamanho 6: χ(G)=2, tempo=0.0000s ✓
  [7/48] Tamanho 7: χ(G)=4, tempo=0.0000s ✓
  [8/48] Tamanho 7: χ(G)=3, tempo=0.0000s ✓
  [9/48] Tamanho 7: χ(G)=2, tempo=0.0000s ✓
  [10/48] Tamanho 8: χ(G)=4, tempo=0.0001s ✓
  [11/48] Tamanho 8: χ(G)=4, tempo=0.0001s ✓
  [12/48] Tamanho 8: χ(G)=3, tempo=0.0000s ✓
  [13/48] Tamanho 9: χ(G)=4, tempo=0.0000s ✓
  [14/48] Tamanho 9: χ(G)=4, tempo=0.0000s ✓
  [15/48] Tamanho 9: χ(G)=3, tempo=0.0000s ✓
  [16/48] Tamanho 10: χ(G

{'forca_bruta':     id_instancia  tamanho  num_vertices  num_arestas  num_cores  \
 0              1        5             5            1          2   
 1              2        5             5            1          2   
 2              3        5             5            3          2   
 3              4        6             6            4          2   
 4              5        6             6            3          2   
 5              6        6             6            3          2   
 6              7        7             7           13          4   
 7              8        7             7           11          3   
 8              9        7             7            3          2   
 9             10        8             8           14          4   
 10            11        8             8           12          4   
 11            12        8             8            7          3   
 12            13        9             9           19          4   
 13            14        9       